<a href="https://colab.research.google.com/github/ZzwahwahzZ/Conquer-Mobile-Assets./blob/main/notebooks/Conquer_Mobile_Color_Texture_Test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Conquer Mobile — اختبار الألوان والخامات (3 مجسمات)

دفتر اختبار مستقل وآمن لمرحلة ما بعد TripoSR. لا يعيد توليد المجسمات ولا يغيّر الملفات الأصلية. يقرأ أرشيف المجسمات من Google Drive، ويقرأ الصور الأصلية من مستودع GitHub، ثم ينشئ نسخ GLB تجريبية ملوّنة في مجلد منفصل.

**تنبيه جودة:** هذه النسخة الأولى تنقل ألوان الصورة إلى رؤوس المجسم (Vertex Colors) بالإسقاط الأمامي؛ وهي تجربة لفحص اتجاه المجسم وتوزيع اللون، وليست بعدُ خامة UV/PBR نهائية. سنراجع النتائج قبل اعتماد طريقة إنتاج الخامات النهائية.

شغّل الخلايا بالترتيب. لا تستخدم Run all قبل اختيار GPU/Runtime المناسب، ولا تحتاج إلى أوزان TripoSR.


In [ ]:
from pathlib import Path
import os, sys, subprocess, zipfile, json, re, shutil
def pip_install(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])
pip_install('trimesh>=4.4.0', 'Pillow>=10', 'numpy>=1.26')
import numpy as np
import trimesh
from PIL import Image
from google.colab import drive
drive.mount('/content/drive')
print('البيئة جاهزة. لا يوجد تنزيل لأوزان TripoSR في هذا الدفتر.')


In [ ]:
# جلب الصور الأصلية فقط من GitHub، ثم العثور على أرشيف المجسمات المحفوظ في Drive.
ASSET_REPO = Path('/content/Conquer-Mobile-Assets')
if not ASSET_REPO.exists():
    subprocess.check_call(['git','clone','--depth','1','https://github.com/ZzwahwahzZ/Conquer-Mobile-Assets..git',str(ASSET_REPO)])
IMAGE_ROOT = ASSET_REPO / 'original_images'
assert IMAGE_ROOT.exists(), f'مجلد الصور الأصلية غير موجود: {IMAGE_ROOT}'
image_files = sorted([p for p in IMAGE_ROOT.rglob('*') if p.suffix.lower() in {'.png','.jpg','.jpeg','.webp'}])
assert image_files, 'لم يتم العثور على الصور الأصلية.'
drive_root = Path('/content/drive/MyDrive')
zip_candidates = list(drive_root.rglob('Conquer_TripoSR_42_Models.zip'))
if not zip_candidates:
    raise FileNotFoundError('لم أجد Conquer_TripoSR_42_Models.zip داخل Google Drive. تأكد أن الأرشيف موجود في MyDrive.')
ZIP_PATH = zip_candidates[0]
WORK = Path('/content/Conquer_Color_Texture_Test')
RAW = WORK / 'raw_models'
OUT = WORK / 'colored_test'
RAW.mkdir(parents=True, exist_ok=True); OUT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as zf:
    members = [n for n in zf.namelist() if n.lower().endswith('.glb')]
    assert members, 'الأرشيف لا يحتوي ملفات GLB.'
    for n in members: zf.extract(n, RAW)
glb_files = sorted(RAW.rglob('*.glb'))
print('الأرشيف:', ZIP_PATH)
print('الصور الأصلية:', len(image_files), '| مجسمات GLB:', len(glb_files))
print('أول 12 صورة:')
for i,p in enumerate(image_files[:12]): print(i, p.relative_to(IMAGE_ROOT))
print('أول 12 مجسم:')
for i,p in enumerate(glb_files[:12]): print(i, p.name)


## اختيار 3 عينات
اكتب أرقام الصور والمجسمات المتطابقة من القائمتين أعلاه. ابدأ بشخصية، ثم وحش، ثم معدّة/عنصر إن كانت متاحة. لا تفترض أن رقم الصورة يساوي رقم المجسم؛ طابق الاسم قبل التشغيل.


In [ ]:
# عدّل هذه القوائم بعد مراجعة الاسمين في القائمتين أعلاه.
# كل عنصر: (رقم الصورة الأصلية، رقم ملف GLB المقابل)
SAMPLES = [(0, 0), (1, 1), (2, 2)]
assert len(SAMPLES) == 3, 'اختر ثلاث عينات فقط للاختبار الأول.'
for image_idx, mesh_idx in SAMPLES:
    assert 0 <= image_idx < len(image_files) and 0 <= mesh_idx < len(glb_files)
    print('IMAGE:', image_files[image_idx].relative_to(IMAGE_ROOT), '<-> GLB:', glb_files[mesh_idx].name)


In [ ]:
def colorize_by_front_projection(mesh_path, image_path, out_path, flip_x=False, flip_y=False, image_size=1024):
    mesh = trimesh.load(mesh_path, force='mesh', process=False)
    if not isinstance(mesh, trimesh.Trimesh) or len(mesh.vertices) == 0 or len(mesh.faces) == 0:
        raise ValueError(f'هندسة غير صالحة: {mesh_path}')
    im = Image.open(image_path).convert('RGB')
    im.thumbnail((image_size, image_size))
    rgb = np.asarray(im)
    v = np.asarray(mesh.vertices, dtype=np.float64)
    lo, hi = v.min(axis=0), v.max(axis=0)
    span = np.maximum(hi - lo, 1e-8)
    # إسقاط أمامي أولي: X إلى عرض الصورة وY إلى ارتفاعها.
    u = (v[:,0] - lo[0]) / span[0]
    vv = (hi[1] - v[:,1]) / span[1]
    if flip_x: u = 1.0 - u
    if flip_y: vv = 1.0 - vv
    px = np.clip(np.rint(u * (rgb.shape[1]-1)).astype(int), 0, rgb.shape[1]-1)
    py = np.clip(np.rint(vv * (rgb.shape[0]-1)).astype(int), 0, rgb.shape[0]-1)
    colors = rgb[py, px]
    # عتامة كاملة؛ يحافظ GLB على هندسة المجسم كما هي.
    rgba = np.concatenate([colors, np.full((len(colors),1),255,dtype=np.uint8)], axis=1)
    mesh.visual = trimesh.visual.ColorVisuals(mesh=mesh, vertex_colors=rgba)
    mesh.export(out_path, file_type='glb')
    check = trimesh.load(out_path, force='mesh', process=False)
    assert len(check.vertices) and len(check.faces), 'فشل التحقق من GLB الناتج.'
    return len(mesh.vertices), len(mesh.faces), Path(out_path).stat().st_size

report=[]
for n,(ii,mi) in enumerate(SAMPLES, start=1):
    src_img=image_files[ii]; src_mesh=glb_files[mi]
    dest=OUT / f'test_{n:02d}_{src_mesh.stem}_VERTEX_COLOR.glb'
    verts,faces,size=colorize_by_front_projection(src_mesh,src_img,dest)
    report.append({'sample':n,'image':str(src_img.relative_to(IMAGE_ROOT)),'source_glb':src_mesh.name,'output_glb':dest.name,'vertices':verts,'faces':faces,'bytes':size,'status':'SUCCESS'})
    print(f'[{n}/3] SUCCESS | {dest.name} | vertices={verts:,} faces={faces:,} size={size:,} bytes')

import csv
report_path=OUT/'color_test_report.csv'
with report_path.open('w',newline='',encoding='utf-8-sig') as f:
    w=csv.DictWriter(f,fieldnames=list(report[0].keys())); w.writeheader(); w.writerows(report)
print('التقرير:',report_path)


In [ ]:
from IPython.display import display
from pathlib import Path
for p in sorted(OUT.glob('*.glb')):
    print(p.name, '|', p.stat().st_size, 'bytes')
print('المخرجات منفصلة عن الأصول الأصلية:', OUT)


In [ ]:
# ضغط نتائج الاختبار ونسخها إلى Google Drive بمجلد مستقل.
import shutil
archive=shutil.make_archive('/content/Conquer_Color_Texture_Test_3_Samples','zip',root_dir=WORK,base_dir='colored_test')
DEST=drive_root/'Conquer_Color_Texture_Test_3_Samples.zip'
shutil.copy2(archive,DEST)
print('تم حفظ نسخة الاختبار على Drive:',DEST)
print('ملاحظة: لم يتم تعديل أو استبدال أي ملف من مجسمات TripoSR الأصلية.')
